# CharXiv Pipeline Demo and Feature Added Value, Checkpoint 2

This notebook demonstrates the reusable preprocessing pipeline in `src/features/` and answers the core
Checkpoint 2 question, whether the twelve DeLeAn demand dimensions add ROC-AUC over a cheap
metadata-only baseline for each target model.

It uses paper-grouped cross-validation (`StratifiedGroupKFold`, 5 folds by 3 repeats, grouped by
`paperid`), the failure-as-positive convention, one classifier per target, and the training items only.
The result table is written to `results/feature_ablation.csv`.

## 1. Setup and the pipeline

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src.features.preprocessing as P
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
RESULTS = ROOT / "results"

TARGETS = P.TARGET_MODELS
con = P.connect(); train_ids, test_ids = P.get_split(con)
df = P.make_design_matrix(con, train_ids)      # one row per item
con.close()

# Demonstrate fit/transform of the default pipeline (features are shared across targets)
pre = P.build_preprocessor().fit(df)
Xt = pre.transform(df)
print("default pipeline -> matrix", Xt.shape)
print("features:", list(pre.get_feature_names_out()))

default pipeline -> matrix (800, 11)
features: ['demand__VL', 'demand__AS', 'demand__MCr', 'demand__MCu', 'demand__MA', 'demand__VO', 'demand__AT', 'demand__GS', 'demand__QLl', 'demand__KNf', 'demand__QLq']


## 2. Feature configurations

Each configuration toggles the pipeline across three feature sets: metadata (the baseline), the twelve
demand dimensions, and the demand dimensions plus metadata. The same configurations are evaluated for
every target.

In [2]:
CONFIGS = {
    "Metadata (baseline)": dict(include_demand_dims=False, include_metadata=True),
    "Demand dims":         dict(include_demand_dims=True,  include_metadata=False),
    "Demand + metadata":   dict(include_demand_dims=True,  include_metadata=True),
}
LEARNERS = {
    "LogReg": lambda: LogisticRegression(max_iter=1000),
    "RF":     lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                             random_state=0, n_jobs=1),
}

def cv(target, cfg, make_clf, n_splits=5, n_repeats=3):
    X, y, g = df, df[P.ycol(target)].to_numpy(), df.groups.to_numpy()
    aucs, baccs = [], []
    for r in range(n_repeats):
        for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=r).split(X, y, g):
            pipe = Pipeline([("pre", P.build_preprocessor(**cfg)), ("clf", make_clf())])
            pipe.fit(X.iloc[tr], y[tr])
            p = pipe.predict_proba(X.iloc[te])[:, 1]
            aucs.append(roc_auc_score(y[te], p))
            baccs.append(balanced_accuracy_score(y[te], (p >= 0.5).astype(int)))
    return np.mean(aucs), np.std(aucs), np.mean(baccs)

## 3. Added-value ablation

In [3]:
rows = []
for target in TARGETS:
    for cname, cfg in CONFIGS.items():
        for lname, mk in LEARNERS.items():
            a, s, b = cv(target, cfg, mk)
            rows.append({"target": target, "config": cname, "learner": lname,
                         "roc_auc": round(a, 4), "roc_auc_sd": round(s, 4), "bal_acc": round(b, 4)})
    print("done:", target)
ablation = pd.DataFrame(rows)
ablation.to_csv(RESULTS / "feature_ablation.csv", index=False)
print("saved -> results/feature_ablation.csv")

done: GPT-4o


done: Claude-3-5-Sonnet


done: GPT-4o-Random
saved -> results/feature_ablation.csv


In [4]:
# Pivot for readability: LogReg ROC-AUC by config x target
piv = ablation[ablation.learner == "LogReg"].pivot_table(index="config", columns="target", values="roc_auc")
piv = piv.reindex(list(CONFIGS))[TARGETS]
display(piv.round(4))

target,GPT-4o,Claude-3-5-Sonnet,GPT-4o-Random
config,,,
Metadata (baseline),0.5279,0.5359,0.7267
Demand dims,0.6758,0.6275,0.5799
Demand + metadata,0.6678,0.6170,0.7237


## 4. Does demand add over the metadata baseline?

In [5]:
def auc(target, cfg, learner="LogReg"):
    r = ablation[(ablation.target == target) & (ablation.config == cfg) & (ablation.learner == learner)]
    return float(r.roc_auc.iloc[0])

for t in TARGETS:
    base, dims, both = auc(t, "Metadata (baseline)"), auc(t, "Demand dims"), auc(t, "Demand + metadata")
    print(f"{t:>18}: metadata {base:.3f} | demand {dims:.3f} (Δ{dims-base:+.3f}) | "
          f"demand+meta {both:.3f}")

            GPT-4o: metadata 0.528 | demand 0.676 (Δ+0.148) | demand+meta 0.668
 Claude-3-5-Sonnet: metadata 0.536 | demand 0.627 (Δ+0.092) | demand+meta 0.617
     GPT-4o-Random: metadata 0.727 | demand 0.580 (Δ-0.147) | demand+meta 0.724


## 5. Conclusion and pipeline defaults

For the two real models the demand-dimensions configuration clearly beats the metadata-only baseline.
This is the headline Checkpoint 2 result, the value of the DeLeAn annotation over cheap metadata. For
GPT-4o-Random the demand dimensions do not beat metadata, because a random answer's success is governed
by the answer format rather than the reasoning demand, which is the expected behavior of the negative
control.

Adding metadata on top of the demand dimensions does not help, and notebook 13 already flagged it for
dropping, so `build_preprocessor` defaults to `include_metadata=False`. The defaults set from this
ablation are `include_demand_dims=True`, `include_metadata=False`, and `drop_dims=('CL',)`.

On leakage discipline, the preprocessing is fit inside each cross-validation fold, and the 200-item test
set is never read.

In [6]:
assert set(df.item_id).isdisjoint(set(test_ids)), "test items leaked into the design matrix!"
print("OK: the 200 test ids never enter any training matrix.")

OK: the 200 test ids never enter any training matrix.
